In [0]:
# Databricks notebook source

# =====================================================================
# CELL 1: CONFIGURATION & DATA LOADING (BRONZE LAYER)
# =====================================================================
# Reading from the Delta table created by 01_Gold_Feature_Engineering

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

# ==============================
# READ FROM DELTA TABLE
# ==============================
TABLE_NAME = "workspace.default.krishi_credit_engine_data"

df_raw = spark.read.table(TABLE_NAME)

print(f"✅ Loaded {df_raw.count()} rows from {TABLE_NAME}")
print(f"📊 Columns: {df_raw.columns}")
display(df_raw.limit(5))

# COMMAND ----------

# =====================================================================
# CELL 2: FEATURE ENGINEERING (SILVER LAYER)
# =====================================================================
# This cell adds the SHG/FPO features and encodes categorical columns
# into numeric values that the scoring engine can use.

df = df_raw

# ------------------------------------
# 2A. Farmer Tier Classification
# ------------------------------------
df = df.withColumn("farmer_tier",
    F.when(F.col("land_area_hectares") >= 4.0, "progressive")
     .when(F.col("land_area_hectares") >= 1.0, "mid_tier")
     .otherwise("subsistence")
)

# ------------------------------------
# 2B. SHG/FPO Membership (if not already in data)
# ------------------------------------
if "is_fpo_shg_member" not in df.columns:
    df = df.withColumn("is_fpo_shg_member",
        F.when(F.col("farmer_tier") == "progressive", F.when(F.rand() < 0.85, 1).otherwise(0))
         .when(F.col("farmer_tier") == "mid_tier",    F.when(F.rand() < 0.55, 1).otherwise(0))
         .otherwise(                                    F.when(F.rand() < 0.20, 1).otherwise(0))
    )

# ------------------------------------
# 2C. Loan Repayment History (if not already in data)
# ------------------------------------
if "fpo_shg_loan_repayment_history" not in df.columns:
    rand_col = F.rand()
    df = df.withColumn("fpo_shg_loan_repayment_history",
        F.when(F.col("is_fpo_shg_member") == 0, "Not Applicable")
         .when(rand_col < 0.40, "No Loan Taken")
         .when(
            ((F.col("farmer_tier") == "subsistence") & (rand_col > 0.80)) |
            ((F.col("farmer_tier") == "mid_tier")    & (rand_col > 0.90)) |
            ((F.col("farmer_tier") == "progressive") & (rand_col > 0.98)),
            "Defaulted"
         )
         .otherwise("Repaid On Time")
    )

# ------------------------------------
# 2D. Encode Repayment History to Numeric Score
# ------------------------------------
df = df.withColumn("repayment_score",
    F.when(F.col("fpo_shg_loan_repayment_history") == "Repaid On Time",  100)
     .when(F.col("fpo_shg_loan_repayment_history") == "No Loan Taken",   50)
     .when(F.col("fpo_shg_loan_repayment_history") == "Not Applicable",  30)
     .when(F.col("fpo_shg_loan_repayment_history") == "Defaulted",        0)
     .otherwise(30)
)

print("✅ Feature Engineering Complete")
display(df.groupBy("farmer_tier", "fpo_shg_loan_repayment_history").count().orderBy("farmer_tier"))

# COMMAND ----------

# =====================================================================
# CELL 3: KRISHI CREDIT SCORE CALCULATION
# =====================================================================
# Updated weights as per your requirements:
#   - Telecom Recharge Streak: REDUCED (max ~30 pts)
#   - FASTag Trips:            REDUCED (max ~20 pts)
#   - SHG/FPO Membership:     NEW     (+25 pts)
#   - SHG Repayment on Time:  NEW     (+80 pts bonus / -150 penalty)

# Detect the telecom streak column name (varies across datasets)
telecom_col = "telecom_recharge_streak_months" if "telecom_recharge_streak_months" in df.columns else "telecom_recharge_streak"

df_scored = df.withColumn(
    "krishi_credit_score",
    F.round(
        F.lit(400)                                                          # Base Score
        + (F.col(telecom_col) * 1.5)                                       # Telecom Streak   (max ~30 pts)
        + (F.col("pmkisan_installments_ytd") * 15)                         # PM-KISAN          (max ~45 pts)
        + (F.col("fastag_trips_last_6m") * 1)                             # FASTag Trips      (max ~20 pts)
        + (F.col("pmfby_insurance_enrolled") * 30)                         # Crop Insurance    (max  30 pts)
        + (F.col("land_record_verified") * 40)                             # Land Verified     (max  40 pts)
        + (F.col("upi_merchant_vs_p2p_ratio") * 20)                       # UPI Ratio         (max  20 pts)
        + (F.col("aadhaar_ekyc_verified") * 50)                            # Aadhaar eKYC      (max  50 pts)
        + (F.col("aadhaar_address_matches_mandi") * 20)                    # Mandi Match       (max  20 pts)
        + (F.col("agristack_id_linked") * 30)                              # AgriStack         (max  30 pts)
        + (F.col("is_fpo_shg_member") * 25)                               # SHG Membership    (max  25 pts)
        + F.when(F.col("fpo_shg_loan_repayment_history") == "Repaid On Time", 80)   # Repaid bonus
         .when(F.col("fpo_shg_loan_repayment_history") == "Defaulted", -150)        # Default penalty
         .otherwise(0)
        , 0
    )
)

# Cap the score between 300 and 900
df_scored = df_scored.withColumn("krishi_credit_score",
    F.when(F.col("krishi_credit_score") > 900, 900)
     .when(F.col("krishi_credit_score") < 300, 300)
     .otherwise(F.col("krishi_credit_score"))
)

print("✅ Credit Score Calculated")
print("\n📊 Score Distribution:")
display(df_scored.select("krishi_credit_score").summary())

# COMMAND ----------

# =====================================================================
# CELL 4: LENDING LIMIT CALCULATION
# =====================================================================
# Heavy weightage on:
#   - UPI Activity (50%): Digital cash flow = repayment capacity
#   - Assets / Land (30%): Collateral value
#   - Credit Score (20%): Risk adjustment multiplier
#
# Formula:
#   Base Capacity = (Land Value * 0.30) + (UPI Annualized * 0.50) + (Score Bonus * 0.20)
#   Final Limit   = Base Capacity * Risk Multiplier (from credit score)

df_final = df_scored.withColumn(
    "asset_value_inr",
    # Verified land = ₹1,00,000 per hectare; Unverified = ₹25,000 (discounted)
    F.when(F.col("land_record_verified") == 1, F.col("land_area_hectares") * 100000)
     .otherwise(F.col("land_area_hectares") * 25000)
)

df_final = df_final.withColumn(
    "upi_annual_cashflow",
    # Annualize the monthly UPI average (12 months)
    F.col("upi_monthly_avg_inr") * 12
)

df_final = df_final.withColumn(
    "risk_multiplier",
    # Score 900 -> 1.0x, Score 600 -> 0.5x, Score 300 -> 0.0x
    F.greatest(F.lit(0.0), (F.col("krishi_credit_score") - 300) / 600)
)

df_final = df_final.withColumn(
    "raw_lending_capacity",
    (F.col("asset_value_inr") * 0.30)       # 30% weight to land/assets
    + (F.col("upi_annual_cashflow") * 0.50)  # 50% weight to UPI cash flow
    + (F.col("krishi_credit_score") * 50 * 0.20)  # 20% weight to score bonus
)

df_final = df_final.withColumn(
    "lending_limit_inr",
    # Apply the risk multiplier and round to nearest ₹5,000
    F.round(F.col("raw_lending_capacity") * F.col("risk_multiplier") / 5000, 0) * 5000
)

# Hard rule: Score below 400 = NO lending
df_final = df_final.withColumn("lending_limit_inr",
    F.when(F.col("krishi_credit_score") < 400, 0)
     .otherwise(F.col("lending_limit_inr"))
)

# Hard rule: Defaulters get max ₹10,000 (micro-loan only)
df_final = df_final.withColumn("lending_limit_inr",
    F.when(
        (F.col("fpo_shg_loan_repayment_history") == "Defaulted") & (F.col("lending_limit_inr") > 10000),
        10000
    ).otherwise(F.col("lending_limit_inr"))
)

# Ensure non-negative
df_final = df_final.withColumn("lending_limit_inr",
    F.greatest(F.lit(0), F.col("lending_limit_inr"))
)

print("✅ Lending Limit Calculated")
print("\n💰 Lending Limit Distribution:")
display(df_final.select("lending_limit_inr").summary())

# COMMAND ----------

# =====================================================================
# CELL 5: SAVE TO DELTA LAKE (SILVER TABLE)
# =====================================================================
# This saves the scored + lending data as a permanent Delta table
# that can be queried from SQL Editor and Dashboards.

SILVER_TABLE = "workspace.default.krishi_silver_scored"

# Select the final columns in a clean order
silver_df = df_final.select(
    "user_id",
    "farmer_tier",
    # Financial Signals
    "upi_monthly_avg_inr",
    "upi_transaction_count_30d",
    "upi_merchant_vs_p2p_ratio",
    "upi_annual_cashflow",
    # Asset Signals
    "land_area_hectares",
    "land_record_verified",
    "asset_value_inr",
    # Government & Welfare
    "pmkisan_installments_ytd",
    "avg_recharge_amount_inr",
    telecom_col,
    "pds_ration_withdrawals_6m",
    "fastag_trips_last_6m",
    "pmfby_insurance_enrolled",
    # Identity
    "aadhaar_ekyc_verified",
    "aadhaar_address_matches_mandi",
    "agristack_id_linked",
    # SHG / FPO
    "is_fpo_shg_member",
    "fpo_shg_loan_repayment_history",
    "repayment_score",
    # Outputs
    "krishi_credit_score",
    "risk_multiplier",
    "lending_limit_inr"
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
print(f"✅ Silver table saved: {SILVER_TABLE}")
print(f"📊 Total rows: {silver_df.count()}")

# COMMAND ----------

# =====================================================================
# CELL 6: GENERATIVE AI JUSTIFICATION (GOLD LAYER)
# =====================================================================
# Uses Databricks Foundation Model API (DBRX / Llama 3) to generate
# a human-readable lending justification for each farmer.
#
# NOTE: This requires Databricks Runtime 14.0+ with Foundation Model
#       APIs enabled. If 'ai_query' is not available, the fallback
#       template-based justification will be used instead.

# Read back from Silver
df_gold = spark.read.table(SILVER_TABLE)

# ---- OPTION A: Databricks SQL ai_query (Recommended for production) ----
# Run this in the SQL Editor instead for best performance:
#
#   SELECT 
#     user_id,
#     krishi_credit_score,
#     lending_limit_inr,
#     ai_query(
#       "databricks-meta-llama-3-1-70b-instruct",
#       CONCAT(
#         "You are a rural bank credit officer AI. A farmer (ID: ", user_id,
#         ") has a Krishi Credit Score of ", krishi_credit_score, "/900. ",
#         "Their monthly UPI income is ₹", upi_monthly_avg_inr,
#         ", they own ", land_area_hectares, " hectares of land",
#         " (verified: ", land_record_verified, "). ",
#         "SHG/FPO member: ", is_fpo_shg_member,
#         ", Repayment history: ", fpo_shg_loan_repayment_history, ". ",
#         "We recommend a lending limit of ₹", lending_limit_inr, ". ",
#         "In 2-3 sentences, explain why this amount is safe or risky ",
#         "and what the farmer can do to improve their limit."
#       )
#     ) AS ai_justification
#   FROM workspace.default.krishi_silver_scored
#   WHERE lending_limit_inr > 0
#   LIMIT 100;

# ---- OPTION B: Template-based fallback (works on all runtimes) ----
df_gold = df_gold.withColumn("ai_justification",
    F.concat(
        F.lit("Farmer "), F.col("user_id"), F.lit(" | Score: "), F.col("krishi_credit_score"), F.lit("/900"),
        F.lit(" | Tier: "), F.col("farmer_tier"),
        F.lit(" → Recommended Limit: ₹"), F.col("lending_limit_inr"), F.lit(". "),
        F.lit("Basis: ₹"), F.round(F.col("upi_annual_cashflow"), 0), F.lit(" annual UPI cashflow"),
        F.lit(" + ₹"), F.round(F.col("asset_value_inr"), 0), F.lit(" land asset value. "),
        F.when(F.col("fpo_shg_loan_repayment_history") == "Repaid On Time",
            F.lit("✅ STRONG: SHG loan repaid on time — high trust signal.")
        ).when(F.col("fpo_shg_loan_repayment_history") == "Defaulted",
            F.lit("🚨 RISK: Past SHG default — lending capped at ₹10,000 micro-loan.")
        ).when(F.col("is_fpo_shg_member") == 1,
            F.lit("ℹ️ SHG member with no loan history — moderate trust.")
        ).otherwise(
            F.lit("ℹ️ No SHG/FPO affiliation — score based on digital & asset signals only.")
        )
    )
)

# Save Gold Table
GOLD_TABLE = "workspace.default.krishi_gold_final"
df_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
print(f"✅ Gold table saved: {GOLD_TABLE}")

# Display sample results
print("\n🏆 Sample Lending Recommendations:")
display(df_gold.select("user_id", "farmer_tier", "krishi_credit_score", "lending_limit_inr", "ai_justification").limit(20))

# COMMAND ----------

# =====================================================================
# CELL 7: SUMMARY DASHBOARD DATA
# =====================================================================
# Quick analytics to verify the pipeline output

print("=" * 60)
print("📊 PIPELINE SUMMARY")
print("=" * 60)

# Score distribution by tier
print("\n1️⃣ Average Credit Score by Farmer Tier:")
display(
    df_gold.groupBy("farmer_tier").agg(
        F.count("*").alias("total_farmers"),
        F.round(F.avg("krishi_credit_score"), 0).alias("avg_score"),
        F.round(F.avg("lending_limit_inr"), 0).alias("avg_lending_limit"),
        F.round(F.sum("lending_limit_inr"), 0).alias("total_portfolio_exposure")
    ).orderBy("avg_score")
)

# Repayment history impact
print("\n2️⃣ Lending Limit by Repayment History:")
display(
    df_gold.groupBy("fpo_shg_loan_repayment_history").agg(
        F.count("*").alias("count"),
        F.round(F.avg("krishi_credit_score"), 0).alias("avg_score"),
        F.round(F.avg("lending_limit_inr"), 0).alias("avg_limit")
    ).orderBy("avg_score")
)

# Top 10 safest farmers to lend to
print("\n3️⃣ Top 10 Safest Farmers (Highest Score + Highest Limit):")
display(
    df_gold.filter(F.col("lending_limit_inr") > 0)
    .orderBy(F.col("krishi_credit_score").desc(), F.col("lending_limit_inr").desc())
    .select("user_id", "farmer_tier", "krishi_credit_score", "lending_limit_inr", 
            "upi_monthly_avg_inr", "land_area_hectares", "fpo_shg_loan_repayment_history")
    .limit(10)
)

print("\n✅ Pipeline Complete! Tables ready for Databricks SQL Dashboards.")

✅ Loaded 10001 rows from workspace.default.krishi_credit_engine_data
📊 Columns: ['user_id', 'upi_monthly_avg_inr', 'upi_transaction_count_30d', 'upi_merchant_vs_p2p_ratio', 'land_area_hectares', 'land_record_verified', 'pmkisan_installments_ytd', 'avg_recharge_amount_inr', 'telecom_recharge_streak_months', 'pds_ration_withdrawals_6m', 'fastag_trips_last_6m', 'pmfby_insurance_enrolled', 'aadhaar_ekyc_verified', 'aadhaar_address_matches_mandi', 'agristack_id_linked', 'krishi_credit_score']


user_id,upi_monthly_avg_inr,upi_transaction_count_30d,upi_merchant_vs_p2p_ratio,land_area_hectares,land_record_verified,pmkisan_installments_ytd,avg_recharge_amount_inr,telecom_recharge_streak_months,pds_ration_withdrawals_6m,fastag_trips_last_6m,pmfby_insurance_enrolled,aadhaar_ekyc_verified,aadhaar_address_matches_mandi,agristack_id_linked,krishi_credit_score
102661,28020,16,0.5,1.54,1,2,355,19.0,2.0,0.0,1,1,1,1,705.0
107294,63875,29,0.69,2.27,1,3,399,6.0,4.0,6.0,0,1,1,1,641.0
108298,43374,26,0.77,1.02,1,1,399,14.0,4.0,0.0,0,1,1,1,640.0
109833,15942,22,0.64,2.24,1,1,899,13.0,5.0,18.0,0,0,0,0,569.0
101627,41885,33,0.55,1.86,1,2,355,7.0,4.0,0.0,1,1,0,1,626.0


✅ Feature Engineering Complete


farmer_tier,fpo_shg_loan_repayment_history,count
mid_tier,Repaid On Time,1594
mid_tier,No Loan Taken,1175
mid_tier,Not Applicable,2465
mid_tier,Defaulted,184
progressive,Defaulted,1
progressive,Not Applicable,30
progressive,Repaid On Time,64
progressive,No Loan Taken,68
subsistence,No Loan Taken,382
subsistence,Not Applicable,3518


✅ Credit Score Calculated

📊 Score Distribution:


...


2️⃣ Lending Limit by Repayment History:


fpo_shg_loan_repayment_history,count,avg_score,avg_limit
Defaulted,291,439.0,6529.0
Not Applicable,6013,559.0,99922.0
No Loan Taken,1625,596.0,120542.0
Repaid On Time,2072,675.0,154619.0



3️⃣ Top 10 Safest Farmers (Highest Score + Highest Limit):


user_id,farmer_tier,krishi_credit_score,lending_limit_inr,upi_monthly_avg_inr,land_area_hectares,fpo_shg_loan_repayment_history
103810,mid_tier,781.0,250000.0,38416,2.43,Repaid On Time
101135,mid_tier,781.0,195000.0,22442,3.43,Repaid On Time
105663,mid_tier,775.0,285000.0,38545,3.98,Repaid On Time
109365,mid_tier,774.0,110000.0,8246,2.65,Repaid On Time
100525,mid_tier,772.0,265000.0,48903,1.13,Repaid On Time
109693,mid_tier,771.0,300000.0,50397,2.37,Repaid On Time
103423,mid_tier,771.0,270000.0,45399,2.23,Repaid On Time
107432,mid_tier,771.0,225000.0,37704,1.86,Repaid On Time
100415,mid_tier,770.0,205000.0,27702,2.98,Repaid On Time
106764,mid_tier,770.0,185000.0,23595,2.83,Repaid On Time



✅ Pipeline Complete! Tables ready for Databricks SQL Dashboards.
